# MP16 — Reusable Training Pipeline

Build a small reproducible training run, then understand the existing research pipeline's exact-resume contract. This is an expanded teaching implementation, not evidence of learner mastery.

## Inputs, outputs and assumptions
A dataset maps an index to an input/target pair. A DataLoader groups these pairs into minibatches. Here inputs and targets are float32 tensors shaped `[N, 1]`; the toy target is `y = 2x + 1`, with dimensionless units. The output is a trained linear model, a measured loss history and recovery checks. CPU, one loading process and a fixed seed make this small demonstration deterministic within its recorded environment.

Training updates parameters. Validation compares candidates without gradients. The held-out test stays unused until the protocol and model selection are frozen. Synthetic interpolation points are not independent geographic scenes.

## Mathematics and hand example
For batch B, mean squared error is L = sum((prediction_i - target_i)^2)/|B|. SGD updates each parameter theta to theta - learning_rate * gradient(L). For predictions [1, 3] and targets [2, 1], squared errors are [1, 4], so batch MSE is 2.5. If one batch contains two examples at MSE 2.5 and the final batch one example at MSE 1, epoch MSE is (2*2.5 + 1*1)/3 = 2, not the unweighted average 1.75. Always weight batch means by their example counts.

An epoch visits the dataset once; a step performs one optimizer update. With 16 examples and batch size 4, one epoch has four steps. Our research trainer instead samples with replacement for a specified number of steps; do not call those steps epochs.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import torch
from torch.utils.data import TensorDataset, DataLoader
x = torch.arange(5, dtype=torch.float32)[:, None]
dataset = TensorDataset(x, 2*x+1)
loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0)
shapes = [tuple(a.shape) for a,b in loader]
assert shapes == [(2,1),(2,1),(1,1)]
assert (2*2.5+1*1)/3 == 2
print('Batch shapes:', shapes)

## A tiny overfit with an explicit training specification
Run `python scripts/pipeline_experiment.py` from the repository root. It trains a one-input linear layer on 16 fixed points for 100 epochs using SGD, learning rate 0.1 and four examples per batch. Validation uses 15 different interpolation points and no optimizer steps. This noise-free toy should fit almost exactly; failure indicates a mechanics problem, not a research discovery.

The script executes the same seeded run twice and compares every parameter exactly. It separately uses the production SR trainer for an eight-step run versus a four-step run resumed to step eight. Checkpoint files are temporary; only measured reports are retained.

In [ ]:
report = json.loads((ROOT/'reports/mp16-pipeline.json').read_text())
assert report['regression']['seeded_parameters_equal']
assert report['regression']['final_epoch_mse'] < 1e-10
assert report['production_resume']['parameters_exact']
print(json.dumps(report, indent=2))

## What exact recovery must save
Weights alone reproduce predictions, but do not preserve the next training update. A resumable checkpoint stores model parameters, optimizer moments, the sampling generator, global random state, current step, best validation value and selected weights, configuration and dataset digest. `src/rssr/experiment.py` writes to a temporary file and replaces the destination after serialization. `last.pt` is resumable; `best.pt` is explicitly inference-only and is rejected for resume.

Changing the dataset digest, batch size or optimizer settings changes the experiment. The production trainer rejects those mismatches and allows only a larger target step count. This lesson tests epoch-boundary toy reproducibility and step-boundary production recovery; it does not promise arbitrary multi-worker DataLoader mid-epoch recovery or bitwise equivalence across platforms.

## Real-data connection and limitations
The existing `reports/geographic-study.json` documents six runs with independent scene splits: four training scenes and four validation scenes, three seeds, CNN and hybrid models, 1,000 steps each. Reuse those verified runs rather than silently training again. They use NAIP aerial imagery with synthetic degradation, not paired satellite ground truth. The mean validation PSNR gains were about 1.571 dB for CNN and 1.334 dB for hybrid; the hybrid did not win. The held-out scene remains unused. Source hashes, configuration and license are recorded in the study and dataset manifests.

The synthetic SR recovery fixture intentionally repeats a ramp across splits to isolate mechanics. It cannot measure generalization. Never report its validation loss as research accuracy.

## Practice — try before reading solutions
1. For 10 examples and batch size 4 with `drop_last=False`, list batch sizes and steps per epoch.
2. Explain why saving only weights cannot reproduce Adam's next step.
3. Why is `model.eval()` insufficient to disable gradients? What does `inference_mode()` add?
4. Propose a recovery acceptance test that catches changed data.
5. If the final batch has one example, why can averaging batch means bias the epoch loss?

## Worked solutions
1. Sizes 4, 4, 2; three steps. With `drop_last=True`, two steps and two omitted examples.
2. Adam carries moving averages of gradients and squared gradients, plus update counters. Those affect the next update even with identical weights.
3. Evaluation mode changes layers such as dropout; it does not turn off autograd. `torch.inference_mode()` prevents gradient tracking for evaluation computations.
4. Save a checkpoint, alter one prepared tile, then verify checksum validation rejects the run before an optimizer update. A changed split manifest must also fail its digest check.
5. Each example should have equal weight; an unweighted mean gives the one-example batch as much influence as a full batch.

## Reflection
The exact-resume checks establish a software property on a controlled CPU fixture. They do not establish scientific quality. Next, MP17 investigates convolution itself; retain this training/checkpoint machinery instead of rebuilding it.

References: [PyTorch beginner tutorial](https://pytorch.org/tutorials/beginner/basics/intro.html), [project brief](../stage_04_math_pytorch/MP16_reusable_training_pipeline/README.md).